# Silvertip CTB 50-Year Production Profile — CSV / JupyterLite Version

This notebook is designed for **JupyterLite**, using **CSV inputs only**.

It will:
- match wells **by Well Name only**
- use the Silvertip CTB well list CSV to define included wells
- read actual production and forecast CSV exports
- use **Column A = Well Name, H = Date, I = Oil (BBL/month), J = Gas (MCF/month)** in production and forecast exports
- use actual production through each well's last actual month
- switch to forecast beginning the following month
- build a **600-month (50-year)** profile per well
- aggregate all wells by calendar month
- save output as CSV files

Upload the three CSV files into the same JupyterLite folder as this notebook before running it.

## 1. Import packages

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re

PROFILE_MONTHS = 600

print('Packages loaded successfully.')

## 2. Find and read the three CSV input files

The filenames should contain these words somewhere:
- `silvertip`
- `production`
- `forecast`


In [ ]:
all_files = glob.glob('*.csv')

print('CSV files found:')
for f in all_files:
    print('  ', f)

def find_file(keyword):
    matches = [f for f in all_files if keyword.lower() in os.path.basename(f).lower()]
    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find a CSV file containing '{keyword}' in the filename."
        )
    if len(matches) > 1:
        print(f"\nMultiple files found for '{keyword}':")
        for f in matches:
            print('  ', f)
        print('Using:', matches[0])
    return matches[0]

well_list_file = find_file('silvertip')
production_file = find_file('production')
forecast_file = find_file('forecast')

print('\nUsing:')
print('Well list: ', well_list_file)
print('Production:', production_file)
print('Forecast:  ', forecast_file)

well_list_raw = pd.read_csv(well_list_file)
production_raw = pd.read_csv(production_file)
forecast_raw = pd.read_csv(forecast_file)

print('\nFiles loaded successfully.')

## 3. Helper functions

In [ ]:
def normalize_well_name(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r'\s+', ' ', value)
    return value.upper()

def parse_date(series):
    parsed = pd.to_datetime(series, errors='coerce')
    return parsed.dt.to_period('M').dt.to_timestamp()


## 4. Prepare the Silvertip CTB well list

In [ ]:
well_name_candidates = [
    c for c in well_list_raw.columns
    if 'well' in str(c).lower() and 'name' in str(c).lower()
]

if len(well_name_candidates) > 0:
    well_list_name_col = well_name_candidates[0]
elif well_list_raw.shape[1] >= 3:
    well_list_name_col = well_list_raw.columns[2]
else:
    raise ValueError('Could not determine the Well Name column in the Silvertip CTB well list.')

print('Using CTB Well Name column:', well_list_name_col)

ctb_wells = well_list_raw[[well_list_name_col]].copy()
ctb_wells.columns = ['Well_Name']
ctb_wells['Well_Name'] = ctb_wells['Well_Name'].astype(str).str.strip()
ctb_wells = ctb_wells[
    ctb_wells['Well_Name'].notna()
    & (ctb_wells['Well_Name'] != '')
    & (ctb_wells['Well_Name'].str.lower() != 'nan')
]
ctb_wells['Match_Name'] = ctb_wells['Well_Name'].apply(normalize_well_name)
ctb_wells = ctb_wells.drop_duplicates('Match_Name').reset_index(drop=True)

print('CTB wells found:', len(ctb_wells))
display(ctb_wells.head())

## 5. Prepare production and forecast data

This uses the physical CSV column positions you specified:
- A = Well Name
- H = Date
- I = Oil BBL/month
- J = Gas MCF/month

In [ ]:
if production_raw.shape[1] < 10:
    raise ValueError('Production CSV does not contain at least columns A-J.')

production = production_raw.iloc[:, [0, 7, 8, 9]].copy()
production.columns = ['Well_Name', 'Date', 'Oil_BBL', 'Gas_MCF']
production['Match_Name'] = production['Well_Name'].apply(normalize_well_name)
production['Date'] = parse_date(production['Date'])
production['Oil_BBL'] = pd.to_numeric(production['Oil_BBL'], errors='coerce').fillna(0)
production['Gas_MCF'] = pd.to_numeric(production['Gas_MCF'], errors='coerce').fillna(0)
production = production[
    production['Match_Name'].notna() & production['Date'].notna()
].copy()
production = (
    production.groupby(['Match_Name', 'Date'], as_index=False)[['Oil_BBL', 'Gas_MCF']]
    .sum()
)

if forecast_raw.shape[1] < 10:
    raise ValueError('Forecast CSV does not contain at least columns A-J.')

forecast = forecast_raw.iloc[:, [0, 7, 8, 9]].copy()
forecast.columns = ['Well_Name', 'Date', 'Oil_BBL', 'Gas_MCF']
forecast['Match_Name'] = forecast['Well_Name'].apply(normalize_well_name)
forecast['Date'] = parse_date(forecast['Date'])
forecast['Oil_BBL'] = pd.to_numeric(forecast['Oil_BBL'], errors='coerce').fillna(0)
forecast['Gas_MCF'] = pd.to_numeric(forecast['Gas_MCF'], errors='coerce').fillna(0)
forecast = forecast[
    forecast['Match_Name'].notna() & forecast['Date'].notna()
].copy()
forecast = (
    forecast.groupby(['Match_Name', 'Date'], as_index=False)[['Oil_BBL', 'Gas_MCF']]
    .sum()
)

print(f'Production rows after cleanup: {len(production):,}')
print(f'Forecast rows after cleanup: {len(forecast):,}')

## 6. Build 50-year profile for each well

In [ ]:
all_well_profiles = []
qa_rows = []

for _, well_row in ctb_wells.iterrows():
    original_name = well_row['Well_Name']
    match_name = well_row['Match_Name']

    prod_well = production[production['Match_Name'] == match_name].copy().sort_values('Date')
    fcst_well = forecast[forecast['Match_Name'] == match_name].copy().sort_values('Date')

    found_prod = len(prod_well) > 0
    found_fcst = len(fcst_well) > 0

    if found_prod:
        profile_start = prod_well['Date'].min()
    elif found_fcst:
        profile_start = fcst_well['Date'].min()
    else:
        qa_rows.append({
            'Well_Name': original_name,
            'Found_Production': False,
            'Found_Forecast': False,
            'Profile_Start': pd.NaT,
            'Last_Actual_Month': pd.NaT,
            'First_Forecast_Month_Used': pd.NaT,
            'Final_Profile_Months': 0,
            'Oil_BBL_50yr': 0,
            'Gas_MCF_50yr': 0,
            'QA_Status': 'MISSING FROM BOTH FILES'
        })
        continue

    monthly_dates = pd.date_range(start=profile_start, periods=PROFILE_MONTHS, freq='MS')
    profile = pd.DataFrame({'Date': monthly_dates})
    profile['Well_Name'] = original_name

    last_actual = prod_well['Date'].max() if found_prod else pd.NaT

    prod_temp = prod_well[['Date', 'Oil_BBL', 'Gas_MCF']].rename(columns={
        'Oil_BBL': 'Actual_Oil_BBL',
        'Gas_MCF': 'Actual_Gas_MCF'
    })
    profile = profile.merge(prod_temp, on='Date', how='left')

    fcst_temp = fcst_well[['Date', 'Oil_BBL', 'Gas_MCF']].rename(columns={
        'Oil_BBL': 'Forecast_Oil_BBL',
        'Gas_MCF': 'Forecast_Gas_MCF'
    })
    profile = profile.merge(fcst_temp, on='Date', how='left')

    if found_prod:
        actual_mask = profile['Date'] <= last_actual
        forecast_mask = profile['Date'] > last_actual
    else:
        actual_mask = pd.Series(False, index=profile.index)
        forecast_mask = pd.Series(True, index=profile.index)

    profile['Oil_BBL'] = 0.0
    profile['Gas_MCF'] = 0.0
    profile['Data_Source'] = ''

    profile.loc[actual_mask, 'Oil_BBL'] = profile.loc[actual_mask, 'Actual_Oil_BBL'].fillna(0)
    profile.loc[actual_mask, 'Gas_MCF'] = profile.loc[actual_mask, 'Actual_Gas_MCF'].fillna(0)
    profile.loc[actual_mask, 'Data_Source'] = 'Actual'

    profile.loc[forecast_mask, 'Oil_BBL'] = profile.loc[forecast_mask, 'Forecast_Oil_BBL'].fillna(0)
    profile.loc[forecast_mask, 'Gas_MCF'] = profile.loc[forecast_mask, 'Forecast_Gas_MCF'].fillna(0)
    profile.loc[forecast_mask, 'Data_Source'] = 'Forecast'

    first_forecast_used = last_actual + pd.offsets.MonthBegin(1) if found_prod else profile_start

    missing_fcst_after_actual = False
    if found_prod:
        required_dates = set(profile.loc[profile['Date'] > last_actual, 'Date'])
        fcst_dates_available = set(fcst_well['Date'])
        if not required_dates.issubset(fcst_dates_available):
            missing_fcst_after_actual = True

    if not found_fcst:
        status = 'NO FORECAST FOUND'
    elif missing_fcst_after_actual:
        status = 'FORECAST DOES NOT COVER FULL 50 YEARS'
    elif not found_prod:
        status = 'NO ACTUALS - FORECAST ONLY'
    else:
        status = 'OK'

    qa_rows.append({
        'Well_Name': original_name,
        'Found_Production': found_prod,
        'Found_Forecast': found_fcst,
        'Profile_Start': profile_start,
        'Last_Actual_Month': last_actual,
        'First_Forecast_Month_Used': first_forecast_used,
        'Final_Profile_Months': len(profile),
        'Oil_BBL_50yr': profile['Oil_BBL'].sum(),
        'Gas_MCF_50yr': profile['Gas_MCF'].sum(),
        'QA_Status': status
    })

    all_well_profiles.append(
        profile[['Well_Name', 'Date', 'Data_Source', 'Oil_BBL', 'Gas_MCF']]
    )

if len(all_well_profiles) == 0:
    raise ValueError('No CTB wells could be matched to production or forecast data.')

well_level = pd.concat(all_well_profiles, ignore_index=True)
qa = pd.DataFrame(qa_rows).sort_values(['QA_Status', 'Well_Name']).reset_index(drop=True)

print('Well-level profile built successfully.')

## 7. Aggregate all wells by calendar month

In [ ]:
aggregated = (
    well_level.groupby('Date', as_index=False)[['Oil_BBL', 'Gas_MCF']]
    .sum()
    .sort_values('Date')
)

aggregated['Cum_Oil_BBL'] = aggregated['Oil_BBL'].cumsum()
aggregated['Cum_Gas_MCF'] = aggregated['Gas_MCF'].cumsum()
aggregated['Days_In_Month'] = aggregated['Date'].dt.days_in_month
aggregated['Oil_BPD'] = aggregated['Oil_BBL'] / aggregated['Days_In_Month']
aggregated['Gas_MCFD'] = aggregated['Gas_MCF'] / aggregated['Days_In_Month']

summary = pd.DataFrame({
    'Metric': [
        'CTB wells in well list',
        'Wells with production match',
        'Wells with forecast match',
        'Wells included in final profile',
        'Wells with QA status OK',
        'Aggregated first month',
        'Aggregated last month',
        'Total Oil BBL',
        'Total Gas MCF'
    ],
    'Value': [
        len(ctb_wells),
        int(qa['Found_Production'].sum()),
        int(qa['Found_Forecast'].sum()),
        well_level['Well_Name'].nunique(),
        int((qa['QA_Status'] == 'OK').sum()),
        aggregated['Date'].min(),
        aggregated['Date'].max(),
        aggregated['Oil_BBL'].sum(),
        aggregated['Gas_MCF'].sum()
    ]
})

display(aggregated.head(12))

## 8. Save CSV outputs

This creates four CSV files that you can download directly from JupyterLite.

In [ ]:
aggregated.to_csv('Silvertip_CTB_Aggregated_50yr.csv', index=False)
well_level.to_csv('Silvertip_CTB_Well_Level_50yr.csv', index=False)
qa.to_csv('Silvertip_CTB_QA.csv', index=False)
summary.to_csv('Silvertip_CTB_Summary.csv', index=False)

print('Created:')
print('  Silvertip_CTB_Aggregated_50yr.csv')
print('  Silvertip_CTB_Well_Level_50yr.csv')
print('  Silvertip_CTB_QA.csv')
print('  Silvertip_CTB_Summary.csv')

## 9. Review QA issues

In [ ]:
qa[qa['QA_Status'] != 'OK']